# Pushout Construction: Finite HaPPY Toy (d=2, N=6)

This notebook constructs the finite pushout algebra:
$$P = (B \oplus C) / \operatorname{span}\{(\alpha(a), -\beta(a)) : a \in A\}$$

**Setup:**
- $A = M_2(\mathbb{C})$ (logical algebra, dimension 4)
- $B = M_6(\mathbb{C})$ (boundary algebra, dimension 36)
- $C = M_2(\mathbb{C})$ (bulk copy, dimension 4)
- $\alpha: A \to B$ embeds into top-left $2 \times 2$ block
- $\beta: A \to C$ is the identity map

**Output:** Left-multiplication matrices for the 36-dimensional quotient algebra

In [ ]:
import numpy as np
from numpy.linalg import svd, norm
import matplotlib.pyplot as plt

# Utilities
def vec(mat):
    "Column-major vectorization (Fortran order) of a matrix."
    return mat.reshape(-1, order='F')

def mat_from_vec(v, rows, cols):
    return v.reshape((rows, cols), order='F')

In [ ]:
d = 2      # logical dimension
N = 6      # boundary dimension
b = 2      # bulk dimension

dim_A = d*d
dim_B = N*N
dim_C = b*b
V_space_dim = dim_B + dim_C

def matrix_units(d):
    mats = []
    for i in range(d):
        for j in range(d):
            M = np.zeros((d, d), dtype=complex)
            M[i, j] = 1.0
            mats.append(M)
    return mats

E_A = matrix_units(d)  # basis of A (matrix units)
print("Dims: A={}, B={}, C={}, V_space={}".format(dim_A, dim_B, dim_C, V_space_dim))

In [ ]:
# Isometry V embedding logical 2-dim into first two coords of 6-dim
V = np.zeros((N, d), dtype=complex)
V[0, 0] = 1.0
V[1, 1] = 1.0
assert np.allclose(V.conj().T @ V, np.eye(d))

def alpha(O):
    # embed O into top-left block of N x N matrix
    M = np.zeros((N, N), dtype=complex)
    M[:d, :d] = O
    return M

def beta(O):
    # identity copy into M_b
    return O.copy()

In [ ]:
R_cols = []
for E in E_A:
    vB = vec(alpha(E))
    vC = vec(beta(E))
    r = np.concatenate([vB, -vC])
    R_cols.append(r.reshape(-1,1))

R = np.hstack(R_cols)   # shape (V_space_dim, dim_A)
print("R shape:", R.shape)

In [ ]:
U, S, Vh = svd(R, full_matrices=True)
tol = 1e-12
rankR = (S > tol).sum()
q_dim = V_space_dim - rankR
print("rank(R) =", rankR, ", quotient dimension q_dim =", q_dim)

# compute nullspace basis for R^T using SVD of R^T
Ut, St, Vht = svd(R.T, full_matrices=True)
rankRt = (St > tol).sum()
nullspace = Vht.T[:, rankRt:]   # columns are nullspace vectors for R^T
Q_basis = nullspace  # shape (V_space_dim, q_dim)
print("Q_basis shape:", Q_basis.shape)

In [ ]:
def project_to_quotient(v):
    # projection via orthonormal columns of Q_basis
    return Q_basis.conj().T @ v

def representative_from_coords(coords):
    return Q_basis @ coords

# representative vectors for quotient basis (columns of Q_basis)
rep_vectors = [Q_basis[:, i] for i in range(Q_basis.shape[1])]
q_dim = len(rep_vectors)
print("Number of representative vectors:", q_dim)

In [ ]:
def multiply_Vspace(u, v):
    # u, v in V_space concatenated vec(B) || vec(C)
    vecB_u = u[:dim_B]; vecC_u = u[dim_B:]
    vecB_v = v[:dim_B]; vecC_v = v[dim_B:]
    B_u = mat_from_vec(vecB_u, N, N); C_u = mat_from_vec(vecC_u, b, b)
    B_v = mat_from_vec(vecB_v, N, N); C_v = mat_from_vec(vecC_v, b, b)
    prodB = B_u @ B_v; prodC = C_u @ C_v
    return np.concatenate([vec(prodB), vec(prodC)])

# Build left-multiplication matrices L_i
L_mats = []
for i in range(q_dim):
    rep_i = rep_vectors[i]
    cols = []
    for j in range(q_dim):
        rep_j = rep_vectors[j]
        prod = multiply_Vspace(rep_i, rep_j)
        coords_prod = project_to_quotient(prod)
        cols.append(coords_prod)
    L = np.column_stack(cols)
    L_mats.append(L)

print("Built {} left-multiplication matrices of shape {}x{}".format(len(L_mats), q_dim, q_dim))

In [ ]:
def multiply_coords(coords_u, coords_v):
    u = representative_from_coords(coords_u)
    v = representative_from_coords(coords_v)
    prod = multiply_Vspace(u, v)
    return project_to_quotient(prod)

# pick random triples
rng = np.random.default_rng(12345)
for trial in range(5):
    x = rng.normal(size=(q_dim,)) + 1j*rng.normal(size=(q_dim,))
    y = rng.normal(size=(q_dim,)) + 1j*rng.normal(size=(q_dim,))
    z = rng.normal(size=(q_dim,)) + 1j*rng.normal(size=(q_dim,))
    left = multiply_coords(multiply_coords(x,y), z)
    right = multiply_coords(x, multiply_coords(y,z))
    diff_norm = norm(left - right)
    print(f"trial {trial}: associativity error norm = {diff_norm:.3e}")

In [ ]:
coords_basis0 = np.eye(q_dim)[:,0]
M0 = np.zeros((q_dim, q_dim), dtype=complex)
for idx in range(q_dim):
    M0 += coords_basis0[idx] * L_mats[idx]

eigvals = np.linalg.eigvals(M0)
plt.figure(figsize=(5,3))
plt.hist(np.real(eigvals), bins=20, alpha=0.7)
plt.title("Histogram of Real parts of eigenvalues (left action of basis 0)")
plt.xlabel("Re(eig)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
try:
    import sympy as sp
    print("SymPy available; running small exact check for R rank.")
    # Build rational matrix for R by converting complex entries to exact rationals when possible
    R_sym = sp.Matrix(R.tolist())
    rank_R_sym = R_sym.rank()
    print("Exact rank(R) [SymPy] = ", rank_R_sym)
except Exception as e:
    print("SymPy exact check skipped or failed:", e)